##v0.3 Objective
Metadata-Aware Retrieval

New features:
✅ Page-aware document loading
✅ Metadata-rich chunks
✅ Chunk IDs
✅ Source attribution
✅ FAISS with metadata
✅ Citation-ready retrieval

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00


In [2]:
import os
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 2 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
- SurendaranathK_Presentation.pdf


In [5]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 2 document(s).


In [6]:
def chunk_text(text, chunk_size=1000, overlap=200):

    paragraphs = text.split("\n\n")

    chunks = []

    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= chunk_size:

            current_chunk += paragraph + "\n\n"

        else:

            chunks.append(current_chunk.strip())

            current_chunk = current_chunk[-overlap:] + paragraph + "\n\n"

    if current_chunk:

        chunks.append(current_chunk.strip())

    return chunks

In [7]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        chunks = chunk_text(page["text"])

        for chunk in chunks:

            all_chunks.append(
                {
                    "chunk_id": chunk_id,
                    "document": document["filename"],
                    "page": page["page"],
                    "text": chunk
                }
            )

            chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 60 chunks.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(60, 384)


In [10]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

60


In [11]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    results = []

    for idx, score in zip(indices[0], distances[0]):

        results.append(
            {
                "chunk_id": all_chunks[idx]["chunk_id"],
                "document": all_chunks[idx]["document"],
                "page": all_chunks[idx]["page"],
                "score": float(score),
                "text": all_chunks[idx]["text"]
            }
        )

    return results

In [18]:
results = retrieve(
    "What algorithm was used for classification?",
    top_k=5
)

for r in results:

    print("=" * 90)

    print(f"Document   : {r['document']}")
    print(f"Page       : {r['page']}")
    print(f"Chunk ID   : {r['chunk_id']}")
    print(f"Similarity : {r['score']:.4f}")

    print("-" * 90)

    print(r["text"][:500])

    print()

Document   : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page       : 8
Chunk ID   : 15
Similarity : 0.4022
------------------------------------------------------------------------------------------
3.4 Data Splitting and Validation
For each classifier, the available simulation dataset was partitioned into a training set (90%
of the data) and a validation set (10%) using stratified sampling to preserve the relative class
proportions. This ensures that each class including minority classes such as prompt collapse
events is adequately represented in both subsets.
To further avoid bias from an “unusually easy” or “unusually difficult” validation set, the
difficulty-aware splitting strategy des

Document   : SurendaranathK_Presentation.pdf
Page       : 5
Chunk ID   : 51
Similarity : 0.3962
------------------------------------------------------------------------------------------
Input: inspiral GW parameters (Mtot, q, Λ̃ , χ eff) from NR simulations.
Classification: Gradie

In [13]:
faiss.write_index(
    index,
    "knowledgehub.index"
)

In [14]:
loaded_index = faiss.read_index(
    "knowledgehub.index"
)

print(loaded_index.ntotal)

60


In [15]:
import time

query = "What algorithm was used for classification?"

start = time.time()

retrieve(query)

end = time.time()

print(f"FAISS Retrieval Time: {end-start:.6f} seconds")

FAISS Retrieval Time: 0.025916 seconds


In [19]:
query = "Summarize the machine learning model used."

results = retrieve(query, top_k=3)

print("=" * 90)
print("Retrieved Sources")
print("=" * 90)

for i, r in enumerate(results, 1):

    print()

    print(f"Source {i}")

    print(f"Document   : {r['document']}")
    print(f"Page       : {r['page']}")
    print(f"Chunk ID   : {r['chunk_id']}")
    print(f"Similarity : {r['score']:.4f}")

    print("-" * 90)

Retrieved Sources

Source 1
Document   : SurendaranathK_Presentation.pdf
Page       : 1
Chunk ID   : 47
Similarity : 0.4114
------------------------------------------------------------------------------------------

Source 2
Document   : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page       : 22
Chunk ID   : 42
Similarity : 0.3799
------------------------------------------------------------------------------------------

Source 3
Document   : SurendaranathK_Presentation.pdf
Page       : 5
Chunk ID   : 51
Similarity : 0.3736
------------------------------------------------------------------------------------------
